In [ ]:
# Full Colab driver script (paste into Colab cell and run)
# - installs pinned libs
# - uploads repo zip
# - extracts repo
# - installs sitecustomize override for HF-backed LLM client (lazy imports)
# - builds cache if missing (with corrected cache_dir handling)
# - runs diagnostics (cache files, sqlite count, FAISS load, exact/semantic tests)
# - runs run_batch and downloads output.csv


# 0) Clean + install pins
!pip -q install "transformers==4.43.3" "accelerate==0.33.0" "sentence-transformers==2.5.1" sentencepiece "pandas==2.2.2" torch faiss-cpu simhash z3-solver sympy


import os, sys, shutil, subprocess, zipfile, tarfile, json, time
from pathlib import Path


os.environ["SENTENCE_TRANSFORMERS_NO_TRAINING"] = "1"


PATCH_DIR = Path("/content/patch")
REPO_DIR  = Path("/content/repo")
DATA_DIR  = Path("/content/repo/data")
CACHE_DIR = Path("/content/repo/cache")
for p in [PATCH_DIR, CACHE_DIR / "faiss", CACHE_DIR / "sqlite"]:
    p.mkdir(parents=True, exist_ok=True)


# 1) Upload repo and extract
from google.colab import files
print("Upload the repo archive (.zip/.tar.gz/.tgz/.tar).")
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No repo archive uploaded.")
archive = next((n for n in uploaded if n.lower().endswith((".zip",".tar.gz",".tgz",".tar"))), None)
if not archive:
    raise SystemExit("Please upload a .zip or .tar(.gz/.tgz) archive of the repo.")


TMP_EXTRACT = Path("/content/tmp_repo")
shutil.rmtree(TMP_EXTRACT, ignore_errors=True)
TMP_EXTRACT.mkdir(parents=True, exist_ok=True)


def extract_archive(path, dest):
    low = path.name.lower()
    if low.endswith(".zip"):
        with zipfile.ZipFile(path, 'r') as zf:
            zf.extractall(dest)
    else:
        mode = "r:gz" if low.endswith((".tar.gz",".tgz")) else "r:"
        with tarfile.open(path, mode) as tf:
            tf.extractall(dest)


extract_archive(Path(archive), TMP_EXTRACT)


def find_repo_root(base: Path) -> Path:
    for p in base.rglob("src"):
        if (p / "agent").exists() and (p / "inference").exists() and (p / "skill_cache").exists():
            return p.parent
    raise FileNotFoundError("Could not find src/agent, src/inference, and src/skill_cache in the uploaded archive.")


found_root = find_repo_root(TMP_EXTRACT)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.move(str(found_root), str(REPO_DIR))


# ===== Ensure repo importability in the current process =====
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# also set PYTHONPATH for subprocesses
os.environ["PYTHONPATH"] = f"{PATCH_DIR}:{REPO_DIR}:{os.environ.get('PYTHONPATH','')}"
print("Inserted repo to sys.path and set PYTHONPATH:", os.environ["PYTHONPATH"])
# ===== end sys.path fix =====


# 2) sitecustomize override:
# NOTE: heavy ML imports (torch/transformers) are placed inside _build_client_cls()
# so that sitecustomize does not error at import time in subprocesses that don't yet have packages.
sitecustomize_code = r'''
# sitecustomize.py in PATCH_DIR — lazy HF-backed OllamaClient override
import sys, importlib, json, re
from typing import Optional, Dict, Any, List


repo_root = "/content/repo"
if repo_root not in sys.path:
    sys.path.append(repo_root)


# set model name in params
try:
    params = importlib.import_module("src.agent.params")
    setattr(params, "LLM_MODEL_NAME", "microsoft/Phi-3.5-mini-instruct")
except Exception:
    # if params import fails, continue; tests will reveal it
    pass


# import target module to override (safe even if it references the real network OO client)
llm_mod = importlib.import_module("src.agent.llm_client")


def _extract_first_json_segment(s: str) -> Optional[str]:
    if not s:
        return None
    s = s.strip()

    # Strategy 1: Find complete JSON with proper depth tracking
    for opener, closer in [('{', '}'), ('[', ']')]:
        start_idx = s.find(opener)
        if start_idx == -1:
            continue
        depth = 0
        for i in range(start_idx, len(s)):
            if s[i] == opener:
                depth += 1
            elif s[i] == closer:
                depth -= 1
                if depth == 0:
                    candidate = s[start_idx:i+1]
                    try:
                        obj = json.loads(candidate)
                        # Validate it's a proper action dict
                        if isinstance(obj, dict) and "action_type" in obj:
                            return candidate
                    except Exception:
                        pass
                    break

    # Strategy 2: Try all JSON-like segments
    matches = re.findall(r'(\{[^{}]*\}|\{[^}]*\{[^}]*\}[^}]*\})', s)
    for seg in matches:
        try:
            obj = json.loads(seg)
            if isinstance(obj, dict):
                return seg
        except Exception:
            pass

    return None



def _build_client_cls():
    # heavy imports only when actually constructing the client class
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch


    model_id = getattr(params, "LLM_MODEL_NAME", "microsoft/Phi-3.5-mini-instruct")


    class HFBackedOllamaClient:
        def __init__(self, model: str = model_id):
            self.model_id = model or model_id
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_id, use_fast=True, trust_remote_code=True)
            # Prefer FP16 for speed; fall back to FP32 as needed
            if torch.cuda.is_available():
                dtype = torch.float16
            else:
                dtype = torch.float32  # CPU fallback
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=dtype, device_map="auto", trust_remote_code=True, attn_implementation="eager"
            )
            if getattr(self.model, "config", None):
                self.model.config.use_cache = True
                self.model.config.attn_implementation = "eager"
            if getattr(self.model, "generation_config", None):
                self.model.generation_config.use_cache = True


        def _chat_prompt(self, system_hint: str, user_prompt: str) -> str:
            sys_msg = system_hint or (
                "You are a strict function caller. When asked for JSON, output ONLY valid JSON with no prose, no code fences."
            )
            messages = [
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": user_prompt},
            ]
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


        def _generate_chat(self, system_hint: str, user_prompt: str, max_tokens: int, temperature: float) -> Optional[str]:
            do_sample = (temperature or 0.0) > 0.0
            prompt = self._chat_prompt(system_hint, user_prompt)
            preview = (prompt or "").replace("\n", " ")
            print(f"[LLM][in] cache=True attn=eager max_new_tokens={max_tokens} temp={temperature} prompt={preview}")
            try:
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
                with torch.inference_mode():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=max_tokens,
                        do_sample=do_sample,
                        temperature=max(temperature, 1e-5),
                        use_cache=True,
                        eos_token_id=self.tokenizer.eos_token_id,
                        pad_token_id=self.tokenizer.eos_token_id,
                        repetition_penalty=1.05,
                    )
                gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
                text = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
                print(f"[LLM][out] {text.replace('\n',' ')}")
                return text
            except Exception as e:
                print(f"[LLM][err] {e}")
                return None


        def generate(self, prompt: str, max_tokens: int = 512, temperature: float = 0.0) -> Optional[str]:
            return self._generate_chat("", prompt, max_tokens=max_tokens, temperature=temperature)


        def generate_json(self, *args, **kwargs) -> Optional[Dict[str, Any]]:
            system_hint = kwargs.pop("system_hint", None)
            user_prompt = kwargs.pop("user_prompt", None)
            schema_text = kwargs.pop("schema_text", None)
            max_tokens = int(kwargs.pop("max_tokens", 512))
            temperature = float(kwargs.pop("temperature", 0.0))


            if system_hint is None and user_prompt is None and schema_text is None:
                if len(args) >= 3:
                    system_hint, user_prompt, schema_text = args[0], args[1], args[2]
                elif len(args) >= 2:
                    prompt_2, schema_2 = args[0], args[1]
                    system_hint = "Output ONLY valid JSON that strictly matches the requested schema. No preamble, no markdown, no code fences."
                    user_prompt = f"{prompt_2}\n\nSchema:\n{schema_2}\nRespond with a single JSON object only."
                    schema_text = schema_2
                else:
                    return None
            else:
                if len(args) >= 1 and user_prompt is None:
                    user_prompt = args[0]
                if len(args) >= 2 and schema_text is None:
                    schema_text = args[1]


            sys_hint = system_hint or "Output ONLY valid JSON that strictly matches the requested schema. No preamble, no markdown, no code fences."
            composed = f"{user_prompt}\n\nSchema:\n{schema_text}\nRespond with a single JSON object only."
            text = self._generate_chat(sys_hint, composed, max_tokens=max_tokens, temperature=0.0)
            if not text:
                return None
            seg = _extract_first_json_segment(text)
            if not seg:
                return None
            try:
                obj = json.loads(seg)
            except Exception:
                return None
            if isinstance(obj, list) and obj and isinstance(obj[0], dict):
                return obj[0]
            if isinstance(obj, dict):
                return obj
            return None


    return HFBackedOllamaClient


# Lazy wrapper with singleton model instance
class LazyOllamaClient:
    _impl_cls = None
    _singleton_instance = None  # ADD THIS LINE

    def __init__(self, *args, **kwargs):
        # Load the class once
        if LazyOllamaClient._impl_cls is None:
            LazyOllamaClient._impl_cls = _build_client_cls()

        # Reuse the SAME model instance for all calls
        if LazyOllamaClient._singleton_instance is None:
            LazyOllamaClient._singleton_instance = LazyOllamaClient._impl_cls(*args, **kwargs)

        self._impl = LazyOllamaClient._singleton_instance  # CHANGED THIS LINE

    def __getattr__(self, name):
        return getattr(self._impl, name)


# attach into repo llm module
setattr(llm_mod, "OllamaClient", LazyOllamaClient)
'''
(PATCH_DIR / "sitecustomize.py").write_text(sitecustomize_code)


# 3) Prefer repo data/cache; else prompt
test_csv = DATA_DIR / "test.csv"
train_csv = DATA_DIR / "train.csv"
skills_db = CACHE_DIR / "sqlite" / "skills.db"
skills_index = CACHE_DIR / "faiss" / "skills.index"


need_upload_data = not test_csv.exists()
need_upload_cache = not (skills_db.exists() and skills_index.exists())
if need_upload_data or need_upload_cache:
    print("Upload any missing files: test.csv, train.csv, skills.db, skills.index (multi-select OK).")
    up_more = files.upload()
    for name in up_more.keys():
        lower = name.lower()
        src = Path(name)
        if lower.endswith("test.csv"):
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(test_csv))
        elif lower.endswith("train.csv"):
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(train_csv))
        elif lower.endswith("skills.db"):
            (CACHE_DIR / "sqlite").mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(skills_db))
        elif lower.endswith("skills.index"):
            (CACHE_DIR / "faiss").mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(skills_index))


print(f"Using:\n- test.csv: {test_csv.exists()} at {test_csv}\n- train.csv: {train_csv.exists()} at {train_csv}\n- skills.db: {skills_db.exists()} at {skills_db}\n- skills.index: {skills_index.exists()} at {skills_index}")


# 4) Build cache if needed (repo builder) -- corrected invocation and more diagnostics
env = os.environ.copy()
# ensure subprocesses use repo code + patch dir
env["PYTHONPATH"] = f"{PATCH_DIR}:{str(REPO_DIR)}:{env.get('PYTHONPATH','')}"
env["PYTHONUNBUFFERED"] = "1"
env["SENTENCE_TRANSFORMERS_NO_TRAINING"] = "1"


need_build = not (skills_db.exists() and skills_index.exists())
if need_build:
    assert train_csv.exists(), "skills.db/index missing and train.csv not available to build the cache."
    print("Building skill cache from train.csv ...")
    # ensure we pass the base cache directory (not a file path) to build_index
    cache_dir_arg = str(CACHE_DIR)
    print("Invoking build_index with cache_dir:", cache_dir_arg)
    p = subprocess.Popen(
        ["python", "-u", "-m", "src.skill_cache.build_index", str(train_csv), cache_dir_arg],
        cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    build_stdout_lines = []
    try:
        for line in p.stdout:
            print(line, end="")
            build_stdout_lines.append(line)
    except Exception as e:
        print("Exception while streaming build output:", e)
    code = p.wait()
    print("build_index exit code:", code)
    if code != 0:
        print("build_index returned non-zero exit code. Last output:")
        print("".join(build_stdout_lines[-40:]))
        raise SystemExit("Cache build failed.")
    else:
        print("build_index finished successfully. Last output lines:")
        print("".join(build_stdout_lines[-20:]))


    # post-build verification
    expected_sqlite = CACHE_DIR / "sqlite" / "skills.db"
    expected_faiss = CACHE_DIR / "faiss" / "skills.index"
    expected_map = CACHE_DIR / "faiss" / "skills_id_map.pkl"
    print("Post-build existence check:")
    print("  expected_sqlite:", expected_sqlite.exists(), expected_sqlite)
    print("  expected_faiss:", expected_faiss.exists(), expected_faiss)
    print("  expected_map:", expected_map.exists(), expected_map)


# ====== DIAGNOSTIC CHECKS ADDED HERE (no other behavior changed) ======
print("\n===== DIAGNOSTIC CHECKS: cache files, sqlite count, FAISS load, exact/semantic tests =====\n")
try:
    # list cache dirs
    print("Cache directory listing:")
    for d in [CACHE_DIR, CACHE_DIR / "sqlite", CACHE_DIR / "faiss"]:
        try:
            print(f"  {d}:")
            for p in sorted([str(x) for x in Path(d).glob("*")]):
                print("    ", p)
        except Exception as e:
            print("    (error listing)", e)


    # check for core files
    sqlite_path = CACHE_DIR / "sqlite" / "skills.db"
    faiss_index_path = CACHE_DIR / "faiss" / "skills.index"
    id_map_path = CACHE_DIR / "faiss" / "skills_id_map.pkl"
    print("\nExpected cache files exist status:")
    print("  skills.db:", sqlite_path.exists(), sqlite_path)
    print("  skills.index:", faiss_index_path.exists(), faiss_index_path)
    print("  skills_id_map.pkl:", id_map_path.exists(), id_map_path)


    # try sqlite count
    try:
        from src.skill_cache.sqlite_store import SQLiteStore
        ss = SQLiteStore(str(sqlite_path)) if sqlite_path.exists() else None
        if ss:
            print("\nSQLiteStore opened. row count:", ss.count())
        else:
            print("\nSQLiteStore not available (skills.db missing).")
    except Exception as e:
        print("\nOpening SQLiteStore failed:", e)


    # try load IndexStore
    try:
        from src.skill_cache.index_store import IndexStore
        if sqlite_path.exists() and faiss_index_path.exists() and id_map_path.exists():
            try:
                idx = IndexStore(str(sqlite_path), cache_dir=str(CACHE_DIR))
                print("IndexStore loaded. FAISS ntotal:", getattr(idx.index, "ntotal", None))
                print("IndexStore id_map sample (first 10):", getattr(idx, "id_map", [])[:10])
            except Exception as e:
                print("IndexStore load failed:", e)
        else:
            print("Skipping IndexStore load (missing files).")
    except Exception as e:
        print("IndexStore import failed:", e)


    # if test.csv exists, run an exact/semantic test for row 0
    try:
        import pandas as pd
        if test_csv.exists():
            df = pd.read_csv(str(test_csv))
            if len(df) > 0:
                test_row = df.iloc[0]
                test_problem = str(test_row.get("problem_statement") or test_row.get("problem_text") or "")
                test_topic = str(test_row.get("topic") or "")
                print("\nTest row 0 preview:")
                print("  topic:", test_topic)
                print("  problem (short):", test_problem[:400])


                # compute SHA and exact lookup
                try:
                    if sqlite_path.exists():
                        sha = SQLiteStore.compute_problem_sha(test_problem)
                        print("  computed SHA:", sha)
                        hit = None
                        try:
                            hit = ss.get_by_problem_sha(sha) if ss else None
                        except Exception as e:
                            print("  exact lookup failed:", e)
                        print("  exact_lookup hit:", bool(hit))
                        if hit:
                            print("  exact hit composed_text (short):", (hit.get("composed_text") or "")[:300])
                            print("  exact hit solution (from DB):", hit.get("solution"))
                    else:
                        print("  No sqlite db to exact lookup against.")
                except Exception as e:
                    print("  SHA / exact lookup exception:", e)


                # semantic retrieval (IndexStore)
                try:
                    if sqlite_path.exists() and faiss_index_path.exists():
                        idx = IndexStore(str(sqlite_path), cache_dir=str(CACHE_DIR))
                        sims = idx.retrieve_similar(test_problem, topic=test_topic, top_k=3)
                        print("\n  retrieve_similar results (top):")
                        if not sims:
                            print("   (no similar results returned; list empty)")
                        for s in sims:
                            rec = s.get("record") if isinstance(s, dict) else s
                            print("   score:", s.get("score"), "id:", (rec.get("id") if rec else None))
                            if rec:
                                print("    composed_text (short):", (rec.get("composed_text") or "")[:300])
                                print("    solution:", rec.get("solution"))
                    else:
                        print("  No FAISS/sqlite files to run semantic retrieval.")
                except Exception as e:
                    print("  retrieve_similar failed:", e)
            else:
                print("test.csv exists but is empty.")
        else:
            print("test.csv not found; skipping test row checks.")
    except Exception as e:
        print("Test row diagnostics failed:", e)


except Exception as e:
    print("Top-level diagnostics block failed:", e)


print("\n===== END DIAGNOSTIC CHECKS =====\n")
# ====== END DIAGNOSTIC CHECKS ======


# 5) Verify override (prevents original HTTP client; uses same CLI objects)
print("\nVerifying LLM override…")
p = subprocess.Popen(
    ["python", "-u", "-c",
     "import importlib; from src.agent.params import LLM_MODEL_NAME; from src.agent.llm_client import OllamaClient; "
     "print('LLM_MODEL_NAME:', LLM_MODEL_NAME); print('OllamaClient module:', OllamaClient.__module__); print('OK:', hasattr(OllamaClient,'generate'))"],
    cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in p.stdout:
    print(line, end="")
p.wait()


# 6) Run with streaming (fast, eager + cache)
out_csv = REPO_DIR / "output.csv"
if out_csv.exists():
    out_csv.unlink()


print("\nRunning batch inference (Phi-3.5 Mini, eager + KV cache, chat-formatted)…\n")
p = subprocess.Popen(
    ["python", "-u", "-m", "src.inference.run_batch", str(test_csv), str(out_csv), "--cache-dir", str(CACHE_DIR)],
    cwd=str(REPO_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
try:
    for line in p.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("Interrupted while streaming run_batch output.")
code = p.wait()
print(f"\nProcess exited with code: {code}")


# 7) Download result
from google.colab import files as colab_files
fallback = REPO_DIR / "output.cs"
target = out_csv if out_csv.exists() else (fallback if fallback.exists() else None)
if target and target.exists():
    print("Downloading:", str(target))
    colab_files.download(str(target))
else:
    print("No output file found at:", out_csv, "or", fallback)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.1/29.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 123.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is 

Saving repo 10 (2).zip to repo 10 (2) (1).zip
Inserted repo to sys.path and set PYTHONPATH: /content/patch:/content/repo:/env/python
Upload any missing files: test.csv, train.csv, skills.db, skills.index (multi-select OK).


Using:
- test.csv: True at /content/repo/data/test.csv
- train.csv: True at /content/repo/data/train.csv
- skills.db: False at /content/repo/cache/sqlite/skills.db
- skills.index: False at /content/repo/cache/faiss/skills.index
Building skill cache from train.csv ...
Invoking build_index with cache_dir: /content/repo/cache
build_index exit code: 0
build_index finished successfully. Last output lines:

Post-build existence check:
  expected_sqlite: False /content/repo/cache/sqlite/skills.db
  expected_faiss: False /content/repo/cache/faiss/skills.index
  expected_map: False /content/repo/cache/faiss/skills_id_map.pkl

===== DIAGNOSTIC CHECKS: cache files, sqlite count, FAISS load, exact/semantic tests =====

Cache directory listing:
  /content/repo/cache:
     /content/repo/cache/.DS_Store
     /content/repo/cache/faiss
     /content/repo/cache/sqlite
  /content/repo/cache/sqlite:
     /content/repo/cache/sqlite/.DS_Store
  /content/repo/cache/faiss:
     /content/repo/cache/faiss/.DS_S

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>